Código actividad Tensor de Inercia, autovalores y autovectores

Luis Enrique Niño Aza y Juan Esteban Gómez

In [28]:
#Importar las librerías a utilizar
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from mpl_toolkits.mplot3d import Axes3D

# === 1. LEER ARCHIVO CSV ===
df = pd.read_csv("datosmasas.csv")
masas = df["masas"].to_numpy()
pos = df[["x", "y", "z"]].to_numpy()

#Calculo masa total (1533 masas)
masa_total = np.sum(masas)

# === 2. FUNCIONES ===
#Posición centro de masa
def centro_de_masa(masas, posiciones):
    return np.sum(masas[:, None] * posiciones, axis=0) / np.sum(masas)
#Calculo momento de inercia 2D (matriz simétrica)
def tensor_inercia_2D(masas, posiciones, cm):
    Ixx = np.sum(masas * (posiciones[:, 1] - cm[1])**2)
    Iyy = np.sum(masas * (posiciones[:, 0] - cm[0])**2)
    Ixy = -np.sum(masas * (posiciones[:, 0] - cm[0]) * (posiciones[:, 1] - cm[1]))
    return np.array([[Ixx, Ixy],
                     [Ixy, Iyy]])
#Calclo momento de inercia 3d (matriz simétrica)
def tensor_inercia_3D(masas, posiciones, cm):
    I = np.zeros((3, 3))
    for m, r in zip(masas, posiciones - cm):
        x, y, z = r
        I += m * np.array([
            [y**2 + z**2, -x*y, -x*z],
            [-x*y, x**2 + z**2, -y*z],
            [-x*z, -y*z, x**2 + y**2]
        ])
    return I

# === 3. ANÁLISIS 2D === (Cálculo de los 3 momentos)
cm_2d = centro_de_masa(masas, pos[:, :2])
tensor_2d = tensor_inercia_2D(masas, pos[:, :2], cm_2d)
autoval_2d, autovec_2d = np.linalg.eigh(tensor_2d)

print("=== ANÁLISIS EN 2D ===")
print("Centro de masa (x, y):", cm_2d)
print("Tensor de inercia 2D:\n", tensor_2d)
print("Autovalores:", autoval_2d)
print("Autovectores (columnas):\n", autovec_2d)
print("Masa total:", masa_total)

# === 4. ANÁLISIS 3D === (Cálculo de los 3 momentos)
cm_3d = centro_de_masa(masas, pos)
tensor_3d = tensor_inercia_3D(masas, pos, cm_3d)
autoval_3d, autovec_3d = np.linalg.eigh(tensor_3d)

print("\n=== ANÁLISIS EN 3D ===")
print("Centro de masa (x, y, z):", cm_3d)
print("Tensor de inercia 3D:\n", tensor_3d)
print("Autovalores:", autoval_3d)
print("Autovectores (columnas):\n", autovec_3d)

# === 5. GRAFICAR 2D: ejes principales (y guardar imagen) ===
fig, ax = plt.subplots() #Malla de la imagen
ax.set_title("Ejes principales de inercia (2D)")
ax.set_xlabel("x")
ax.set_ylabel("y")

# Distribución de puntos (Grafica las masas)
ax.scatter(pos[:, 0], pos[:, 1], c='blue', label='Partículas')
ax.scatter(*cm_2d, color='red', label='Centro de masa')

# Ejes principales (Autovectores de la matriz de inercia)
for val, vec in zip(autoval_2d, autovec_2d.T):
    start = cm_2d
    end = cm_2d + vec * np.sqrt(val) * 0.05
    ax.plot([start[0], end[0]], [start[1], end[1]], linewidth=2, label="Eje principal") #Forma el vector propio

ax.legend()
ax.axis("equal")
plt.grid()
plt.savefig("inercia_2D.png", dpi=300)
plt.close()

# === 6. GRAFICAR 3D: ejes principales (vista frontal y rotada) ===

# Vista 1: frontal
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
ax.set_title("Ejes principales de inercia (3D) - Vista Frontal")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z")

# Distribución de puntos (Grafica las masas)
ax.scatter(pos[:, 0], pos[:, 1], pos[:, 2], color='blue', label="Partículas")
ax.scatter(*cm_3d, color='red', label="Centro de masa")

#Ejes principales
for val, vec in zip(autoval_3d, autovec_3d.T):
    end = cm_3d + vec * np.sqrt(val) * 0.03
    ax.plot([cm_3d[0], end[0]], [cm_3d[1], end[1]], [cm_3d[2], end[2]], linewidth=2, label="Eje principal")

ax.legend()
plt.savefig("inercia_3D_frontal.png", dpi=300)
plt.close()

# Vista 2: rotada (ángulo diferente)
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
ax.set_title("Ejes principales de inercia (3D)")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z")

# Distribución de puntos (Grafica las masas)
#ax.scatter(pos[:, 0], pos[:, 1], pos[:, 2], color='blue')
ax.scatter(*cm_3d, color='red', label="Centro de masa")

#Ejes principales
for val, vec in zip(autoval_3d, autovec_3d.T):
    end = cm_3d + vec * np.sqrt(val) * 0.03
    ax.plot([cm_3d[0], end[0]], [cm_3d[1], end[1]], [cm_3d[2], end[2]], linewidth=2, label="Eje principal")

ax.legend()
ax.view_init(elev=30, azim=300)  # <- Ángulo rotado
plt.savefig("Ejes principales de inercia (3D).png", dpi=300)
plt.close()

=== ANÁLISIS EN 2D ===
Centro de masa (x, y): [825.81521504 776.91852172]
Tensor de inercia 2D:
 [[ 9.63660148e+08 -9.11747911e+08]
 [-9.11747911e+08  9.58535589e+08]]
Autovalores: [4.93463569e+07 1.87284938e+09]
Autovectores (columnas):
 [[-0.7061125  -0.70809967]
 [-0.70809967  0.7061125 ]]
Masa total: 4627.0

=== ANÁLISIS EN 3D ===
Centro de masa (x, y, z): [825.81521504 776.91852172  15.5033499 ]
Tensor de inercia 3D:
 [[ 1.06550347e+09 -9.11747911e+08  7.14204864e+06]
 [-9.11747911e+08  1.06037891e+09  1.92959724e+06]
 [ 7.14204864e+06  1.92959724e+06  1.92219574e+09]]
Autovalores: [1.51166481e+08 1.92196006e+09 1.97495158e+09]
Autovectores (columnas):
 [[-0.70611307  0.04694255  0.70654139]
 [-0.70808985 -0.05190995 -0.7042117 ]
 [ 0.00361904 -0.99754787  0.06989384]]


In [ ]:
#Diagonalización de la matriz CASO 2D

# Matriz transformada
I_transf = autovec_2d.T @ tensor_2d @ autovec_2d #Producto matricial T*I*T^T

# Redondear la matriz
I_transf_clean = np.round(I_transf, 8)

# Forzar ceros fuera de la diagonal
for i in range(I_transf_clean.shape[0]):
    for j in range(I_transf_clean.shape[1]):
        if i != j:
            I_transf_clean[i, j] = 0.0

# Mostrar resultados
print("Matriz transformada con ceros fuera de la diagonal:")
print(np.round(I_transf_clean, 4))

diag_autoval = np.diag(np.round(autoval_2d, 8)) #Forma diagonal con los autovals.
print("Matriz diagonal de autovalores:")
print(np.round(diag_autoval, 4))

#Prueba de diagonalización
if np.allclose(I_transf_clean, diag_autoval, atol=1e-8):
    print("✅ Confirmado: T^T * I * T = diag(autovalores)")
else:
    print("❌ Algo no coincide, revisar normalización de autovectores.")

Matriz transformada con ceros fuera de la diagonal:
[[4.93463569e+07 0.00000000e+00]
 [0.00000000e+00 1.87284938e+09]]
Matriz diagonal de autovalores:
[[4.93463569e+07 0.00000000e+00]
 [0.00000000e+00 1.87284938e+09]]
✅ Confirmado: T^T * I * T = diag(autovalores)


In [ ]:
#Diagonalización de la matriz CASO 3D

# Matriz transformada
I_transf = autovec_3d.T @ tensor_3d @ autovec_3d #Producto matricial T*I*T^T

# Redondear la matriz
I_transf_clean = np.round(I_transf, 8)

# Forzar ceros fuera de la diagonal
for i in range(I_transf_clean.shape[0]):
    for j in range(I_transf_clean.shape[1]):
        if i != j:
            I_transf_clean[i, j] = 0.0

# Mostrar resultados
print("Matriz transformada con ceros fuera de la diagonal:")
print(np.round(I_transf_clean, 4))

diag_autoval = np.diag(np.round(autoval_3d, 8))#Forma diagonal con los autovals.
print("Matriz diagonal de autovalores:")
print(np.round(diag_autoval, 4))

#Prueba de diagonalización
if np.allclose(I_transf_clean, diag_autoval, atol=1e-8):
    print("✅ Confirmado: T^T * I * T = diag(autovalores)")
else:
    print("❌ Algo no coincide, revisar normalización de autovectores.")

Matriz transformada con ceros fuera de la diagonal:
[[1.51166481e+08 0.00000000e+00 0.00000000e+00]
 [0.00000000e+00 1.92196006e+09 0.00000000e+00]
 [0.00000000e+00 0.00000000e+00 1.97495158e+09]]
Matriz diagonal de autovalores:
[[1.51166481e+08 0.00000000e+00 0.00000000e+00]
 [0.00000000e+00 1.92196006e+09 0.00000000e+00]
 [0.00000000e+00 0.00000000e+00 1.97495158e+09]]
✅ Confirmado: T^T * I * T = diag(autovalores)
